In [2]:
import pandas as pd
from scipy import signal
from scipy import interpolate as interp
import math
import numpy as np
import matplotlib.pyplot as plt
import os
from scipy.signal import butter, filtfilt
import glm_toolbox as glm
import signal_processing_toolbox as processing
import friction_toolbox as fric

plt.close('all')

subjects = ["S01", "S02", "S03", "S04"]
blocks = ["L1", "L2", "L3", "L4", "S1", "S2", "S3", "S4"]

for s in subjects:
    
    # Nombre d'essais dépend du sujet
    if s in ["S01", "S02"]:
        n_trials = 4
    else:
        n_trials = 3

    for b in blocks:
        
        for t in range(1, n_trials + 1):
             # Format 001, 002, ...
            trial = f"{t:03d}"
            
            glm_path = f"./Groupe4_LUNDI_APREM/Gr4_{s}_{b}_{trial}.glm"
            
            print(f"Processing: {glm_path}")
            
            # Vérifie si le fichier existe
            if not os.path.exists(glm_path):
                print("❌ Fichier manquant, skip")
                continue
            
            glm_df = glm.import_data(glm_path)
            # Define baseline, i.e. time interval where no finger forces were applied on the sensors
            baseline = range(0,400)     
            
            # Normal Force exerted by the thumb
            NF_thumb = glm_df.loc[:,'Fygl']-np.nanmean(glm_df.loc[baseline,'Fygl'])
        
            # Vertical Tangential Force exerted by the thumb
            TFx_thumb = glm_df.loc[:,'Fxgl']-np.nanmean(glm_df.loc[baseline,'Fxgl'])
            
            #Horizontal Tangential Force exerted by the thumb
            TFz_thumb = glm_df.loc[:,'Fzgl']-np.nanmean(glm_df.loc[baseline,'Fzgl'])

            # Normal Force exerted by the index
            NF_index = -(glm_df.loc[:,'Fygr']-np.nanmean(glm_df.loc[baseline,'Fygr']))
            
            # Vertical Tangential Force exerted by the index
            TFx_index = glm_df.loc[:,'Fxgr']-np.nanmean(glm_df.loc[baseline,'Fxgr'])
        
            #Horizontal Tangential Force exerted by the index
            TFz_index = glm_df.loc[:,'Fzgr']-np.nanmean(glm_df.loc[baseline,'Fzgr']) 
            
            
            #%% Get acceleration, LF and GF
            time  = glm_df.loc[:,'time'].to_numpy()
            if b in ["L1", "L2", "L3", "L4"]:
                accX_raw  = glm_df.loc[:,'LowAcc_X'].to_numpy()*(-9.81) 
                acc = accX_raw - np.mean(accX_raw[baseline])
            elif b in ["S1", "S2", "S3", "S4"]:
                accZ_raw  = glm_df.loc[:,'LowAcc_Z'].to_numpy()*(-9.81) 
                acc = accZ_raw - np.mean(accZ_raw[baseline])
            GF    = glm_df.loc[:,'GF'].to_numpy()
            GF    = GF-np.nanmean(GF[baseline])
            LFx   = TFx_thumb + TFx_index
            LFz   = TFz_thumb + TFz_index
            LF    = np.hypot(LFx,LFz)
        
            # %%Filter data
            freqAcq=800 #Acquisition sampling frequency
            freqFiltAcc=4 #Low-pass filter cut-off frequency for acceleration signals
            freqFiltForces=30 #Low-pass filter cut-off frequency for force signals
            
            acc = processing.filter_signal(acc, fs = freqAcq, fc = freqFiltAcc)
            GF   = processing.filter_signal(GF,   fs = freqAcq, fc = freqFiltForces)
            LF   = processing.filter_signal(LF,   fs = freqAcq, fc = freqFiltForces)
            LFx  = processing.filter_signal(LFx,  fs = freqAcq, fc = freqFiltForces)
            LFz  = processing.filter_signal(LFz,  fs = freqAcq, fc = freqFiltForces)
            
            
            #%% CUTTING THE TASK INTO SEGMENTS (your first task)
            # For an oscillation task, this is very easy as we just need to find 
            # acceleration peaks
            pk = signal.find_peaks(acc,  prominence=2, distance=1200)
            ipk = pk[0]
            cycle_starts = ipk[:-1]
            cycle_ends = ipk[1:]-1
            
            
            #%% Compute derivative of GF
            dGF = processing.derive(GF,800)
            
            #%% Basic plot of the data
            fig = plt.figure(figsize = [15,7])
            ax  = fig.subplots(3,1)
            
            ax[0].plot(time, acc)
            ax[0].plot(time[ipk],acc[ipk], linestyle='', marker='o', 
                    markerfacecolor='None', markeredgecolor='r')
            ax[0].set_ylabel("Acceleration [m/s^2]", fontsize=13)
            ax[0].set_title(f"GLM data - Sujet {s}, Bloc {b}, Essai {trial}", fontsize=14, fontweight="bold")
            ax[0].set_ylim([-4,4])
            #ax[0].set_xlim([0,60])
            
            # Putting grey patches for movement cycles
            for i in range(0,len(cycle_starts)):
                rect0=plt.Rectangle((time[cycle_starts[i]],ax[0].get_ylim()[0]),\
                                time[cycle_ends[i]-cycle_starts[i]],\
                                ax[0].get_ylim()[1]-ax[0].get_ylim()[0],color='k',alpha=0.3)
                ax[0].add_patch(rect0)
            
            ax[1].plot(time,LF, label="LF")
            ax[1].plot(time,GF, label="GF")
            ax[1].legend(fontsize=12)
            ax[1].set_xlabel("Time [s]", fontsize=13)
            ax[1].set_ylabel("Forces [N]", fontsize=13)
            #ax[1].set_xlim([0,30])
            
            ax[2].plot(time,dGF)
            ax[2].set_xlabel("Time [s]", fontsize=13)
            ax[2].set_ylabel("GF derivative [N/s]", fontsize=13)
            #ax[2].set_xlim([0,30])
        
            output_folder = f"figures/{s}"
            os.makedirs(output_folder, exist_ok=True)

            fig.savefig(f"{output_folder}/{s}_{b}_{trial}.png")
            plt.close(fig)


Processing: ./Groupe4_LUNDI_APREM/Gr4_S01_L1_001.glm
Processing: ./Groupe4_LUNDI_APREM/Gr4_S01_L1_002.glm
Processing: ./Groupe4_LUNDI_APREM/Gr4_S01_L1_003.glm
Processing: ./Groupe4_LUNDI_APREM/Gr4_S01_L1_004.glm
Processing: ./Groupe4_LUNDI_APREM/Gr4_S01_L2_001.glm
Processing: ./Groupe4_LUNDI_APREM/Gr4_S01_L2_002.glm
Processing: ./Groupe4_LUNDI_APREM/Gr4_S01_L2_003.glm
Processing: ./Groupe4_LUNDI_APREM/Gr4_S01_L2_004.glm
Processing: ./Groupe4_LUNDI_APREM/Gr4_S01_L3_001.glm
Processing: ./Groupe4_LUNDI_APREM/Gr4_S01_L3_002.glm
Processing: ./Groupe4_LUNDI_APREM/Gr4_S01_L3_003.glm
Processing: ./Groupe4_LUNDI_APREM/Gr4_S01_L3_004.glm
Processing: ./Groupe4_LUNDI_APREM/Gr4_S01_L4_001.glm
Processing: ./Groupe4_LUNDI_APREM/Gr4_S01_L4_002.glm
Processing: ./Groupe4_LUNDI_APREM/Gr4_S01_L4_003.glm
Processing: ./Groupe4_LUNDI_APREM/Gr4_S01_L4_004.glm
Processing: ./Groupe4_LUNDI_APREM/Gr4_S01_S1_001.glm
Processing: ./Groupe4_LUNDI_APREM/Gr4_S01_S1_002.glm
Processing: ./Groupe4_LUNDI_APREM/Gr4_S01_S1_0

In [3]:
subjects = ["S01", "S02", "S03", "S04"]
trials = 3
friction = "friction"
#%% 
freqAcq=800 # Frequence d'acquisition des donnees
freqFiltForces=20; # Frequence de coupure du filtrage des forces (peut etre modifié)

for s in subjects:
    base_folder = f"figures/{s}"
    friction_folder = f"{base_folder}/friction"

    os.makedirs(friction_folder, exist_ok=True)
    # Initialisation des structures servant à stocker les valeurs de TF/NF et NF
    # aux moments des glissements
    all_mu_points_thumb=[]
    all_NF_thumb=[]
    all_mu_points_index=[]
    all_NF_index=[]
    for t in range(1, trials + 1):
        # Format 001, 002, ...
        trial = f"{t:03d}"
        
        glm_path = f"./Groupe4_LUNDI_APREM/Gr4_{s}_{friction}_{trial}.glm"
        print(f"Processing: {glm_path}")
            
            # Vérifie si le fichier existe
        if not os.path.exists(glm_path):
            print("❌ Fichier manquant, skip")
            continue
        
        glm_df = glm.import_data(glm_path)    
        #%% Extraction des signaux et mise a zero des forces en soustrayant la valeur 
        # moyenne des 500 premieres ms
        baseline = range(0,400)
        
        # Normal Force exerted by the thumb
        NF_thumb = glm_df.loc[:,'Fygl']-np.nanmean(glm_df.loc[baseline,'Fygl'])
        NF_thumb = processing.filter_signal(NF_thumb,fs=freqAcq,fc=freqFiltForces)
        
        # Vertical Tangential Force exerted by the thumb
        TFx_thumb = glm_df.loc[:,'Fxgl']-np.nanmean(glm_df.loc[baseline,'Fxgl'])
        TFx_thumb = processing.filter_signal(TFx_thumb,fs=freqAcq,fc=freqFiltForces)
        
        #Horizontal Tangential Force exerted by the thumb
        TFz_thumb = glm_df.loc[:,'Fzgl']-np.nanmean(glm_df.loc[baseline,'Fzgl'])
        TFz_thumb = processing.filter_signal(TFz_thumb,fs=freqAcq,fc=freqFiltForces)

        # Normal Force exerted by the index
        NF_index = -(glm_df.loc[:,'Fygr']-np.nanmean(glm_df.loc[baseline,'Fygr']))
        NF_index = processing.filter_signal(NF_index,fs=freqAcq,fc=freqFiltForces)
        
        # Vertical Tangential Force exerted by the index
        TFx_index = glm_df.loc[:,'Fxgr']-np.nanmean(glm_df.loc[baseline,'Fxgr'])
        TFx_index = processing.filter_signal(TFx_index,fs=freqAcq,fc=freqFiltForces)
        
        #Horizontal Tangential Force exerted by the index
        TFz_index = glm_df.loc[:,'Fzgr']-np.nanmean(glm_df.loc[baseline,'Fzgr'])
        TFz_index = processing.filter_signal(TFz_index,fs=freqAcq,fc=freqFiltForces)
        
        
        #%%Compute COP manually  
        COPthumb,COPindex = glm.compute_cop(glm_df,baseline)
        
        # Filter CoP
        COPthumb = processing.filter_signal(COPthumb,fs=freqAcq,fc=freqFiltForces)
        COPindex = processing.filter_signal(COPindex,fs=freqAcq,fc=freqFiltForces)
        
        
        #%% Calcul du coefficient de friction au moment du glissement. Voir fonction 
        # find_slip_onset et papier "Barrea et al. 2016" pour plus de détails
        mu_thumb,slip_indexes_thumb,start_search_zones_thumb,end_search_zones_thumb = \
        fric.find_slip_onset(COPthumb,TFx_thumb,TFz_thumb,NF_thumb)
        
        all_mu_points_thumb=[*all_mu_points_thumb,*mu_thumb]
        all_NF_thumb=[*all_NF_thumb,*NF_thumb[slip_indexes_thumb]]
        
        mu_index,slip_indexes_index,start_search_zones_index,end_search_zones_index = \
        fric.find_slip_onset(COPindex,TFx_index,TFz_index,NF_index)
        
        all_mu_points_index=[*all_mu_points_index,*mu_index]
        all_NF_index=[*all_NF_index,*NF_index[slip_indexes_index]]
    
        #%% Graphe de détection du glissement pour l'index
        siz= len(start_search_zones_index)
        time = glm_df.loc[:,'time'].to_numpy()
        fig = plt.figure(figsize = [15,7])
        ax  = fig.subplots(3,1)
        ax[0].set_title("%s - Index" % (glm_path[-14:-4]), fontsize=14, fontweight="bold")
        ax[0].plot(time, COPindex*1000)
        ax[0].set_ylabel("COP [mm]", fontsize=13)
        ax[0].set_ylim([-25,25])
        
        ax[1].plot(time,TFx_index, label="TFv")
        ax[1].plot(time,NF_index, label="NF")
        ax[1].legend(fontsize=12)
        ax[1].set_ylabel("Forces [N]", fontsize=13)
        
        ax[2].plot(time,np.divide(np.hypot(TFx_index,TFz_index),NF_index))
        ax[2].plot(time[slip_indexes_index],mu_index,marker='.',linestyle='')
        ax[2].set_ylim([0,max(mu_index)+0.2])
        ax[2].set_ylabel("TF/NF [-]", fontsize=13)
        ax[2].set_xlabel("Time [s]", fontsize=13)
        
        for i in range(0,siz):
            rect0=plt.Rectangle((time[start_search_zones_index[i]],ax[0].get_ylim()[0]),\
                            time[end_search_zones_index[i]-start_search_zones_index[i]],\
                            ax[0].get_ylim()[1]-ax[0].get_ylim()[0],color='k',alpha=0.2)
            rect1=plt.Rectangle((time[start_search_zones_index[i]],ax[1].get_ylim()[0]),\
                            time[end_search_zones_index[i]-start_search_zones_index[i]],\
                            ax[1].get_ylim()[1]-ax[1].get_ylim()[0],color='k',alpha=0.2)
            rect2=plt.Rectangle((time[start_search_zones_index[i]],ax[2].get_ylim()[0]),\
                            time[end_search_zones_index[i]-start_search_zones_index[i]],\
                            ax[2].get_ylim()[1]-ax[2].get_ylim()[0],color='k',alpha=0.2)
            ax[0].add_patch(rect0)
            ax[1].add_patch(rect1)
            ax[2].add_patch(rect2)
        fig.savefig(f"{friction_folder}/{s}_{trial}_index.png")
        plt.close(fig)
        
        #%% Graphe de détection du moment du glissement pour l'essai "thumb strong"
        siz= len(start_search_zones_thumb)
        time = glm_df.loc[:,'time'].to_numpy()
        fig = plt.figure(figsize = [15,7])
        ax  = fig.subplots(3,1)
        ax[0].set_title("%s - Thumb" % (glm_path[-14:-4]), fontsize=14, fontweight="bold")
        ax[0].plot(time, COPthumb*1000)
        ax[0].set_ylabel("COP [mm]", fontsize=13)
        ax[0].set_ylim([-25,25])
    
        ax[1].plot(time,TFx_thumb, label="TFv")
        ax[1].plot(time,NF_thumb, label="NF")
        ax[1].legend(fontsize=12)
        ax[1].set_ylabel("Forces [N]", fontsize=13)
        
        ax[2].plot(time,np.divide(np.hypot(TFx_thumb,TFz_thumb),NF_thumb))
        ax[2].plot(time[slip_indexes_thumb],mu_thumb,marker='.',linestyle='')
        ax[2].set_ylim([0,max(mu_thumb)+0.2])
        ax[2].set_ylabel("TF/NF [-]", fontsize=13)
        ax[2].set_xlabel("Time [s]", fontsize=13)
        
        for i in range(0,siz):
            rect0=plt.Rectangle((time[start_search_zones_thumb[i]],ax[0].get_ylim()[0]),\
                                time[end_search_zones_thumb[i]-start_search_zones_thumb[i]],\
                                ax[0].get_ylim()[1]-ax[0].get_ylim()[0],color='k',alpha=0.2)
            rect1=plt.Rectangle((time[start_search_zones_thumb[i]],ax[1].get_ylim()[0]),\
                                time[end_search_zones_thumb[i]-start_search_zones_thumb[i]],\
                                ax[1].get_ylim()[1]-ax[1].get_ylim()[0],color='k',alpha=0.2)
            rect2=plt.Rectangle((time[start_search_zones_thumb[i]],ax[2].get_ylim()[0]),\
                                time[end_search_zones_thumb[i]-start_search_zones_thumb[i]],\
                                ax[2].get_ylim()[1]-ax[2].get_ylim()[0],color='k',alpha=0.2)
            ax[0].add_patch(rect0)
            ax[1].add_patch(rect1)
            ax[2].add_patch(rect2)
        
        fig.savefig(f"{friction_folder}/{s}_{trial}_thumb.png")
        plt.close(fig)


    #%% Impression de la valeur des variables dans la console
    #%% Calcul des valeurs de k et n pour l'index
    mu = np.array(all_mu_points_index)
    NF = np.array(all_NF_index)

    mask = (NF > 0) & (mu > 0) & np.isfinite(NF) & np.isfinite(mu)

    mu_clean = mu[mask]
    NF_clean = NF[mask]

    k_index, n_index = fric.get_mu_fit(mu_clean, NF_clean)

    #%% Calcul des valeurs de k et n pour le pouce
    mu = np.array(all_mu_points_thumb)
    NF = np.array(all_NF_thumb)

    mask = (NF > 0) & (mu > 0) & np.isfinite(NF) & np.isfinite(mu)

    mu_clean = mu[mask]
    NF_clean = NF[mask]

    k_thumb, n_thumb = fric.get_mu_fit(mu_clean, NF_clean)

        
    #%% Figure finale pour l'index
    x=np.arange(0.2,30,0.02)
    fig = plt.figure(figsize = [15,7])
    plt.plot(x,k_index*(x**(n_index-1)))
    plt.plot(all_NF_index,all_mu_points_index,linestyle='',marker='.')
    plt.ylim([0,3])
    plt.xlim([0,30])
    plt.title('Coefficient of friction index')
    plt.xlabel('Normal Force [N]')
    plt.ylabel('Static Friction [-]')
    fig.savefig(f"{friction_folder}/{s}_fit_index.png")
    plt.close(fig)

    #%% Figure finale pour le pouce
    x=np.arange(0.2,30,0.02)
    fig = plt.figure(figsize = [15,7])
    plt.plot(x,k_thumb*(x**(n_thumb-1)))
    plt.plot(all_NF_thumb,all_mu_points_thumb,linestyle='',marker='.')
    plt.ylim([0,3])
    plt.xlim([0,30])
    plt.title('Coefficient of friction thumb')
    plt.xlabel('Normal Force [N]')
    fig.savefig(f"{friction_folder}/{s}_fit_thumb.png")
    plt.close(fig)

    print("Index: the value of k is %f and n is %f" %(k_index,n_index))
    print("Thumb: the value of k is %f and n is %f" %(k_thumb,n_thumb))  


Processing: ./Groupe4_LUNDI_APREM/Gr4_S01_friction_001.glm
Processing: ./Groupe4_LUNDI_APREM/Gr4_S01_friction_002.glm
Processing: ./Groupe4_LUNDI_APREM/Gr4_S01_friction_003.glm
Index: the value of k is 1.325693 and n is 0.612304
Thumb: the value of k is 1.554272 and n is 0.596699
Processing: ./Groupe4_LUNDI_APREM/Gr4_S02_friction_001.glm
Processing: ./Groupe4_LUNDI_APREM/Gr4_S02_friction_002.glm
Processing: ./Groupe4_LUNDI_APREM/Gr4_S02_friction_003.glm
Index: the value of k is 1.389559 and n is 0.565049
Thumb: the value of k is 1.668730 and n is 0.540393
Processing: ./Groupe4_LUNDI_APREM/Gr4_S03_friction_001.glm


c:\Users\axelh\OneDrive\Documents\P4_gbio\friction_toolbox.py:65: RuntimeWarning: Mean of empty slice
  slope=np.nanmean(dy_loc[0:np.floor((iStart[i+1]-iStart[i])/2).astype(np.int)])


Processing: ./Groupe4_LUNDI_APREM/Gr4_S03_friction_002.glm
Processing: ./Groupe4_LUNDI_APREM/Gr4_S03_friction_003.glm
Index: the value of k is 1.551897 and n is 0.527030
Thumb: the value of k is 1.418221 and n is 0.622806
Processing: ./Groupe4_LUNDI_APREM/Gr4_S04_friction_001.glm
Processing: ./Groupe4_LUNDI_APREM/Gr4_S04_friction_002.glm
Processing: ./Groupe4_LUNDI_APREM/Gr4_S04_friction_003.glm
Index: the value of k is 1.071801 and n is 0.417466
Thumb: the value of k is 1.539783 and n is 0.464817
